# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [ ]:
# Install Groq SDK (only needed once per Colab session)
!pip install groq -q

In [ ]:
import os, re, json, logging
from groq import Groq

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

# ⚙️  Paste your Groq API key here
GROQ_API_KEY = "your_groq_api_key_here"
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a math expression — stripped to safe chars only."""
    try:
        clean = re.sub(r'[^0-9+\-*/().\s%]', '', expression)
        if not clean.strip():
            return "No valid expression found"
        result = eval(clean)
        return str(round(result, 4))
    except Exception as e:
        logger.error(f"Calculator error: {e}")
        return "Error in calculation"

# Verify
print(calculator("20 + 5 * 3"))   # → 35
print(calculator("100 / 4"))      # → 25.0

In [ ]:
# 🛠️ TOOL 2: Keyword Extractor

STOP_WORDS = {
    "from", "this", "that", "with", "have", "will", "your",
    "what", "when", "where", "which", "their", "there",
    "about", "keywords", "extract", "please"
}

def extract_keywords(text: str) -> list:
    """Deduplicated keyword extraction with stop-word filter."""
    try:
        words = re.findall(r'\b[a-zA-Z]+\b', text)
        seen, keywords = set(), []
        for w in words:
            wl = w.lower()
            if len(wl) > 4 and wl not in STOP_WORDS and wl not in seen:
                seen.add(wl)
                keywords.append(wl)
        return keywords[:5]
    except Exception as e:
        logger.error(f"Keyword extractor error: {e}")
        return []

# Verify
print(extract_keywords("Artificial Intelligence is transforming industries"))
# → ['artificial', 'intelligence', 'transforming', 'industries']

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [ ]:
def groq_response(query: str) -> str:
    """Call Groq LLM for general queries."""
    try:
        response = client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": query}],
            max_tokens=300,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        logger.error(f"Groq API error: {e}")
        return f"LLM Error: {str(e)}"

In [ ]:
INTENT_SIGNALS = {
    "calculation": {
        "calculate": 3, "compute": 3, "evaluate": 3, "solve": 2,
        "plus": 1, "minus": 1, "times": 1, "divided": 1,
        "square": 1, "percent": 1
    },
    "keywords": {
        "keyword": 3, "keywords": 3, "extract": 2,
        "key terms": 3, "important words": 2, "tags": 1
    }
}

def route_intent(query: str) -> str:
    """Score query against each intent; return best match or 'general'."""
    query_lower = query.lower()
    scores = {intent: 0 for intent in INTENT_SIGNALS}

    for intent, signals in INTENT_SIGNALS.items():
        for signal, weight in signals.items():
            if signal in query_lower:
                scores[intent] += weight

    # Digits are strong evidence of a math intent
    if re.search(r'\d', query_lower):
        scores["calculation"] += 2

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "general"

# Verify
print(route_intent("Calculate 20 + 5"))         # → calculation
print(route_intent("Extract keywords from text"))  # → keywords
print(route_intent("What is deep learning?"))      # → general

In [ ]:
def extract_math_expression(query: str) -> str:
    """Pull the numeric expression out of a natural language query."""
    match = re.search(r'[\d\s\+\-\*/\(\)\.%]+', query)
    return match.group().strip() if match else query


def agent(query: str) -> dict:
    """
    Single-agent dispatcher.
    Routes: calculation → calculator | keywords → extractor | else → Groq
    Returns structured JSON-serialisable dict.
    """
    logger.info(f"Query: {query}")
    try:
        intent = route_intent(query)
        logger.info(f"Intent → {intent}")

        if intent == "calculation":
            expr = extract_math_expression(query)
            result = calculator(expr)
            return {"type": "calculation", "expression": expr, "result": result}

        elif intent == "keywords":
            result = extract_keywords(query)
            return {"type": "keywords", "result": result}

        else:
            result = groq_response(query)
            return {"type": "general", "result": result}

    except Exception as e:
        logger.error(f"Agent error: {e}")
        return {"type": "error", "result": str(e)}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [ ]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5 * 3",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Solve 144 / 12",            # tests digit boost in router
    "What is the capital of France?"  # pure general
]

for q in queries:
    print(f"\nQuery : {q}")
    print(f"Output: {json.dumps(agent(q), indent=2)}")
    print("-" * 60)

In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("\nEnter query (type 'exit' to stop): ").strip()
    if user_input.lower() == "exit":
        print("Shutting down agent.")
        break
    if not user_input:
        continue
    print(json.dumps(agent(user_input), indent=2))